# Bayesian hyperparameter optimisation of LightGBM (model 4)

LightGBM was chosen in `trained_models/comparison/`. This notebook tunes it with a Bayesian search (Optuna, TPE sampler) and then
compares the **untuned** model (the baseline, left unchanged in `trained_models/model4/`) with the **tuned** one on the same data, folds and metrics.

Rules: raw candidate table, the 22 selected features, fitting only on `fit_include` rows with `sample_weight`, the 5 stored GroupKFold folds by trip.
`arc_id` is never a feature. The evaluator partition is never read. **Student-test is not used for any tuning decision**: it is evaluated exactly once
per model, after the configuration is frozen (section 4). The search code lives in `tune_lib.py` (same folder), so the study can be resumed from a
terminal with identical code; this notebook runs or resumes it and does every analysis.

In [1]:
"""Setup: paths, the study configuration (all settings live in tune_lib.CONFIG), shared modelling code."""
import sys, json, time, shutil, importlib.util
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, optuna, lightgbm as lgb
HERE = Path.cwd()                                   # trained_models/model4/tuning
sys.path.insert(0, str(HERE)); sys.path.insert(0, str(HERE.parents[1]))
import tune_lib as T
from common.data_loading import SEED, feature_list, load_candidates
from common.experiment import ModelSpec, run_experiment
from common.metrics import reliability_table, expected_calibration_error
FIG = HERE / "figures"; FIG.mkdir(exist_ok=True)
M4 = HERE.parent                                    # trained_models/model4 (untuned baseline, read-only here)
FINAL = HERE.parents[1] / "final_model"
rng = np.random.default_rng(SEED)
(HERE / "config.json").write_text(json.dumps(T.CONFIG, indent=2), encoding="utf-8")
print("optuna", optuna.__version__, "| lightgbm", lgb.__version__, "| features", len(feature_list()))

optuna 5.1.0 | lightgbm 4.7.0 | features 22


## 1. Objective, tie-breaker and procedure

**Objective: maximise the mean 5-fold CV Top-1 on label-in-candidates points.** A point whose label arc is not among its candidates is wrong for
every configuration (the candidate miss rate is fixed by preprocessing), so Top-1 on all points and Top-1 on in-candidate points rank
configurations the same way. The in-candidate version leaves out those fixed misses, which come mostly from three far-heavy trips; they make the
all-points score swing by about 13 points between folds and add noise that has nothing to do with the hyperparameters. CV Top-1 on all points,
normalised log loss and Macro F1 are still recorded for every trial.

**Tie-breaker.** Two trials whose mean scores differ by less than one fold standard error (std / sqrt(5)) cannot be told apart. Among the trials
within one standard error of the best, we take the one with the **lowest normalised log loss**, because the HMM uses the probabilities as emissions.

**Nested early stopping.** In every CV fold, 20% of the fold's *training* trips are held out (trip-grouped split, seed 42, the same split the untuned
model uses) and boosting stops after 100 rounds without improvement of their weighted log loss, with a cap of 2,000 rounds. The CV validation fold is
never used to stop, and student-test is not used at all. The best iteration of every fold is recorded.

**No refit inside the search.** The untuned model refits on all rows of the fold after early stopping. Inside the search we score the early-stopped
model directly: it halves the cost of every trial, and every configuration is treated the same way, so the ranking is fair. The final untuned vs
tuned comparison (section 4) uses the full procedure with refit for both models.

**Sampler and pruner.** TPE (a Bayesian method that models good and bad regions of the search space), seed 42, Optuna's default 10 random start-up
trials. No pruner: every trial keeps complete 5-fold records.

**Budget.** The task asks for 60 trials and at least 40. Low learning rates need over 1,000 rounds, so trials took 1.5-2 minutes on one worker; for the
time we had, the study was cut to **40 unique configurations** and run by **two worker processes in parallel** on the same SQLite study, with 8
LightGBM threads each (`MaxTrialsCallback` stops both workers). Parallel TPE workers propose some points without seeing each other's newest results,
a small loss of sample efficiency.

**A mistake we made and fixed.** The two workers were first started with the same sampler seed, so they proposed identical parameters: trials 5-10
repeat trials 1-3. Those repeated evaluations are kept in the log and flagged (`duplicate_of`); the workers were restarted with seeds 43 and 44, and
the study ran to 46 complete trials = 40 unique configurations. Trials interrupted by the restarts (4, 13, 14) are stored as FAIL. Only COMPLETE trials
are analysed, and each configuration is counted once.

## 2. Search space and why

| Parameter | Range | Why |
|---|---|---|
| `learning_rate` | 0.01-0.2, log | Below 0.01 needs thousands of rounds (past the 2,000 cap); above 0.2 boosting gets coarse. Log scale: equal attention to each factor of 2. |
| `num_leaves` | 15-255, log | Tree complexity. 15 is a shallow tree, 255 is LightGBM's usual upper practical range for 1.8 M rows; the untuned value 63 sits in the middle. |
| `max_depth` | {-1, 4-12} | -1 lets `num_leaves` alone limit the tree (untuned); 4-12 tests whether a depth cap generalises better across trips. |
| `min_child_samples` | 10-300 | Rows per leaf. Small values can memorise single trips; large values smooth. The untuned 100 is inside. |
| `feature_fraction` | 0.5-1.0 | Share of the 22 features per tree; below 0.5 a tree would often miss the dominant distance / heading features. |
| `bagging_fraction` | 0.5-1.0 (freq 1) | Share of rows per round: variance reduction vs information per tree. |
| `lambda_l1` | 1e-3-10, log | L1 on leaf values, from negligible to strong. |
| `lambda_l2` | 1e-3-10, log | L2 on leaf values; the untuned 1.0 is inside. |
| `min_split_gain` | 0-1 | Minimum loss reduction to split; 0 (untuned) allows every useful split. |

Class weighting and sample weights are exactly as in the untuned model (sample weights clean 1.0 / ambiguous 0.5 / far 0.3, no class weights).
**Trial 0 is the untuned configuration** (enqueued), so the baseline is inside the search; the only difference is `lambda_l1` = 0.001 instead of 0,
because 0 is outside a log range.

In [2]:
"""Run or resume the study (SQLite storage). If the 60 complete trials already exist, this cell only loads them."""
t0 = time.time()
study = T.run_study(T.CONFIG["n_trials"])
complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
print(f"complete trials: {len(complete)} | other states: {len(study.trials) - len(complete)} | this cell: {time.time() - t0:.0f}s")

[I 2026-11-18 17:48:46,360] Using an existing study with `study_name='lightgbm_top1_in_cands'` instead of creating a new one.


complete trials: 47 | other states: 3 | this cell: 1s


## 3. Trials, best configuration and the shape of the optimum

In [3]:
"""All trials -> optuna_trials.csv (parameters, every metric, best iterations, time)."""
rows = []
for t in study.trials:
    r = {"trial": t.number, "state": t.state.name, "objective_cv_top1_in_cands": t.value}
    r.update({f"param_{k}": v for k, v in t.params.items()})
    for k, v in t.user_attrs.items():
        if k not in ("per_fold",): r[k] = json.dumps(v) if isinstance(v, list) else v
    r["datetime_start"], r["datetime_complete"] = t.datetime_start, t.datetime_complete
    rows.append(r)
trials = pd.DataFrame(rows)
pcols = [c for c in trials.columns if c.startswith("param_")]
key = trials[pcols].round(10).astype(str).agg("|".join, axis=1)
first = {}; trials["duplicate_of"] = [first.setdefault(k, n) if st == "COMPLETE" else np.nan for k, n, st in zip(key, trials.trial, trials.state)]
trials["duplicate_of"] = [d if (st == "COMPLETE" and d != n) else np.nan for d, n, st in zip(trials.duplicate_of, trials.trial, trials.state)]
trials.to_csv(HERE / "optuna_trials.csv", index=False)
print("duplicate evaluations (same parameters as an earlier trial):", trials.dropna(subset=["duplicate_of"])[["trial", "duplicate_of"]].astype(int).values.tolist())
ok = trials[(trials.state == "COMPLETE") & trials.duplicate_of.isna()].copy()          # each configuration once
total_s = ok.seconds.sum(); wall = (pd.to_datetime(ok.datetime_complete).max() - pd.to_datetime(ok.datetime_start).min()).total_seconds()
print(f"{len(ok)} complete trials | sum of trial times {total_s/60:.1f} min | study wall clock {wall/60:.1f} min | mean {ok.seconds.mean():.0f}s per trial")
chosen, best, se, close = T.select_trial(study)
print(f"best trial #{best.number}: {best.value:.5f} | fold SE {se:.5f} | trials within 1 SE: {[t.number for t in close]}")
print(f"chosen by the tie-break (lowest normalised log loss among them): trial #{chosen.number} ({chosen.value:.5f}, ll_norm {chosen.user_attrs['cv_logloss_norm_mean']:.5f})")
base = study.trials[0]; print(f"trial 0 (untuned): {base.value:.5f}, ll_norm {base.user_attrs['cv_logloss_norm_mean']:.5f}")
cols = ["trial", "objective_cv_top1_in_cands", "cv_top1_in_cands_std", "cv_top1_all_mean", "cv_logloss_norm_mean", "cv_macro_f1_mean", "mean_best_iteration", "seconds"] + [c for c in ok.columns if c.startswith("param_")]
top5 = ok.sort_values("objective_cv_top1_in_cands", ascending=False).head(5)[cols]
display(top5.round(5))
spread = {"best": float(top5.objective_cv_top1_in_cands.iloc[0]), "top5_mean": float(top5.objective_cv_top1_in_cands.mean()),
          "top5_range": float(top5.objective_cv_top1_in_cands.iloc[0] - top5.objective_cv_top1_in_cands.iloc[-1]), "fold_se_of_best": float(se),
          "baseline_trial0": float(base.value), "best_minus_baseline": float(best.value - base.value)}
print("spread of the top 5:", {k: round(v, 5) for k, v in spread.items()})

duplicate evaluations (same parameters as an earlier trial): [[5, 1], [6, 1], [7, 2], [8, 2], [9, 3], [10, 3]]
41 complete trials | sum of trial times 76.3 min | study wall clock 51.5 min | mean 112s per trial
best trial #22: 0.85769 | fold SE 0.01491 | trials within 1 SE: [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
chosen by the tie-break (lowest normalised log loss among them): trial #23 (0.85750, ll_norm 0.42990)
trial 0 (untuned): 0.85521, ll_norm 0.43676


,trial,objective_cv_top1_in_cands,cv_top1_in_cands_std,cv_top1_all_mean,cv_logloss_norm_mean,cv_macro_f1_mean,mean_best_iteration,seconds,param_learning_rate,param_num_leaves,param_max_depth,param_min_child_samples,param_feature_fraction,param_bagging_fraction,param_lambda_l1,param_lambda_l2,param_min_split_gain
22,22,0.85769,0.03334,0.77894,0.43234,0.89592,130.8,103.9,0.06788,215,12,237,0.53000,0.67204,0.00381,0.14927,0.51702
23,23,0.85750,0.03296,0.77891,0.42990,0.89586,339.8,191.0,0.02389,232,12,205,0.54263,0.58246,0.00278,0.23076,0.33468
32,32,0.85719,0.03315,0.77845,0.43414,0.89564,132.8,87.7,0.07283,186,12,203,0.50389,0.60160,0.00320,0.23655,0.25282
48,48,0.85707,0.03244,0.77829,0.43280,0.89556,82.0,60.7,0.08486,204,12,259,0.62074,0.68003,0.00567,0.14116,0.45702
29,29,0.85698,0.03378,0.77845,0.43545,0.89560,435.0,220.3,0.01972,208,12,208,0.55993,0.63579,0.00275,0.81923,0.41085


spread of the top 5: {'best': 0.85769, 'top5_mean': 0.85729, 'top5_range': 0.00071, 'fold_se_of_best': 0.01491, 'baseline_trial0': 0.85521, 'best_minus_baseline': 0.00249}


In [4]:
"""Plots: optimisation history, parameter importance, slice plots of the most important parameters, parallel coordinates."""
import optuna.visualization.matplotlib as ov
import warnings; warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
ax = ov.plot_optimization_history(study); ax.set_title("Optimisation history: CV Top-1 (label in candidates)"); ax.axhline(base.value, c="grey", ls="--", lw=1, label="trial 0 = untuned"); ax.legend(fontsize=8)
plt.gcf().set_size_inches(9, 4); plt.tight_layout(); plt.savefig(FIG / "optimisation_history.png", dpi=140); plt.show()
imp = optuna.importance.get_param_importances(study, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEED))
fig, ax = plt.subplots(figsize=(7, 4)); s = pd.Series(imp).sort_values(); ax.barh(s.index, s.values, color="#2a6f97"); ax.set_xlabel("fANOVA importance (share of objective variance)")
ax.set_title("Hyperparameter importance (fANOVA, seed 42)"); plt.tight_layout(); plt.savefig(FIG / "param_importance.png", dpi=140); plt.show()
top_params = list(pd.Series(imp).sort_values(ascending=False).index[:4]); print("importance:", {k: round(v, 3) for k, v in imp.items()})
axs = ov.plot_slice(study, params=top_params); plt.gcf().set_size_inches(4 * len(top_params), 3.6); plt.tight_layout(); plt.savefig(FIG / "slice_top_params.png", dpi=130); plt.show()
ax = ov.plot_parallel_coordinate(study, params=top_params); plt.gcf().set_size_inches(10, 4.5); plt.tight_layout(); plt.savefig(FIG / "parallel_coordinate.png", dpi=130); plt.show()

C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\3432608471.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.gcf().set_size_inches(9, 4); plt.tight_layout(); plt.savefig(FIG / "optimisation_history.png", dpi=140); plt.show()


C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\3432608471.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Hyperparameter importance (fANOVA, seed 42)"); plt.tight_layout(); plt.savefig(FIG / "param_importance.png", dpi=140); plt.show()
C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\3432608471.py:10: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  axs = ov.plot_slice(study, params=top_params); plt.gcf().set_size_inches(4 * len(top_params), 3.6); plt.tight_layout(); plt.savefig(FIG / "slice_top_params.png", dpi=130); plt.show()


importance: {'max_depth': np.float64(0.4), 'lambda_l1': np.float64(0.229), 'num_leaves': np.float64(0.083), 'min_split_gain': np.float64(0.06), 'lambda_l2': np.float64(0.053), 'min_child_samples': np.float64(0.049), 'learning_rate': np.float64(0.046), 'feature_fraction': np.float64(0.045), 'bagging_fraction': np.float64(0.036)}


C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\3432608471.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axs = ov.plot_slice(study, params=top_params); plt.gcf().set_size_inches(4 * len(top_params), 3.6); plt.tight_layout(); plt.savefig(FIG / "slice_top_params.png", dpi=130); plt.show()


C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\3432608471.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax = ov.plot_parallel_coordinate(study, params=top_params); plt.gcf().set_size_inches(10, 4.5); plt.tight_layout(); plt.savefig(FIG / "parallel_coordinate.png", dpi=130); plt.show()


In [5]:
"""Freeze the chosen configuration -> best_params.json. From here on nothing is changed by any later result."""
best_params = dict(chosen.params)
frozen = {"chosen_trial": chosen.number, "best_trial_by_objective": best.number, "tie_break_rule": T.CONFIG["tie_breaker"],
          "params": best_params, "cv_top1_in_cands_mean": chosen.value, "cv_top1_in_cands_std": chosen.user_attrs["cv_top1_in_cands_std"],
          "cv_logloss_norm_mean": chosen.user_attrs["cv_logloss_norm_mean"], "best_iterations_per_fold": chosen.user_attrs["best_iterations"],
          "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"), "student_test_used_before_freeze": False}
(HERE / "best_params.json").write_text(json.dumps(frozen, indent=2), encoding="utf-8"); print(json.dumps(frozen, indent=1))

{
 "chosen_trial": 23,
 "best_trial_by_objective": 22,
 "tie_break_rule": "trials within one fold standard error (std / sqrt(5)) of the best mean: lowest mean normalised log loss",
 "params": {
  "learning_rate": 0.0238943484631863,
  "num_leaves": 232,
  "max_depth": 12,
  "min_child_samples": 205,
  "feature_fraction": 0.5426347976960108,
  "bagging_fraction": 0.5824633942587291,
  "lambda_l1": 0.0027777042271715075,
  "lambda_l2": 0.23075515131579688,
  "min_split_gain": 0.3346793657763438
 },
 "cv_top1_in_cands_mean": 0.8574973109829418,
 "cv_top1_in_cands_std": 0.032955369512509894,
 "cv_logloss_norm_mean": 0.4299029945366086,
 "best_iterations_per_fold": [
  233,
  409,
  226,
  421,
  410
 ],
 "frozen_at": "2026-11-18 17:48:48",
 "student_test_used_before_freeze": false
}


### Findings

_filled in after the run_

## 4. Untuned vs tuned: same data, same folds, same metrics

Both models go through the **same shared harness** (`common/experiment.py`) the comparison used: 5-fold CV with the stored folds, final fit on all
`fit_include` training rows, then **one** evaluation on student-test. The final fit uses the same procedure as the untuned model: early stopping on a
trip-grouped 20% of the training trips, then a **refit on all training rows for exactly that best iteration** (an inner split, not a mean of the CV
best iterations).

The untuned model is re-run here with model 4's own code (`model4/train_lightgbm.py`) into `untuned_rerun/` so both sides use this machine and
library versions; it is checked against the stored `model4/results.json`, which is left unchanged. The harness does not return out-of-fold
predictions, so a thin wrapper records the predictions the harness asks for (5 CV folds in order, then the training rows, then student) to draw
calibration curves.

In [6]:
"""Specs: untuned = model 4's own fit / predict / save; tuned = the same procedure with the frozen parameters."""
spec4 = importlib.util.spec_from_file_location("train_lightgbm", M4 / "train_lightgbm.py"); m4 = importlib.util.module_from_spec(spec4); spec4.loader.exec_module(m4)
from sklearn.model_selection import GroupShuffleSplit
TUNED_PARAMS = {**T.CONFIG["fixed_params"], **best_params}
def tuned_fit(X, y, w, trip_ids, seed):
    """Early stopping on a trip-grouped 20% of the TRAINING trips, then refit on all rows for best_iteration (as model 4)."""
    params = {**TUNED_PARAMS, "seed": seed, "bagging_seed": seed, "feature_fraction_seed": seed}
    tr, va = next(GroupShuffleSplit(n_splits=1, test_size=T.CONFIG["early_stop_val_share"], random_state=seed).split(X, y, groups=trip_ids))
    feats = feature_list(); rec = {}
    dtr = lgb.Dataset(X[tr], y[tr], weight=w[tr], feature_name=feats, free_raw_data=False)
    dva = lgb.Dataset(X[va], y[va], weight=w[va], reference=dtr, feature_name=feats, free_raw_data=False)
    b = lgb.train(params, dtr, num_boost_round=T.CONFIG["max_rounds"], valid_sets=[dtr, dva], valid_names=["train", "val"],
                  callbacks=[lgb.early_stopping(T.CONFIG["early_stopping_rounds"], verbose=False), lgb.record_evaluation(rec)])
    best_it = int(b.best_iteration)
    hist = {"train": rec["train"]["binary_logloss"], "val": rec["val"]["binary_logloss"], "best_iteration": best_it, "n_val_trips": int(len(np.unique(trip_ids[va])))}
    return lgb.train(params, lgb.Dataset(X, y, weight=w, feature_name=feats), num_boost_round=best_it), hist
def capture(pred_fn, store):
    """Wrap a predict function so every prediction the harness requests is kept (CV folds in order, train rows, student)."""
    def f(model, X):
        p = pred_fn(model, X); store.append(np.asarray(p)); return p
    return f
caps = {"untuned": [], "tuned": []}
specs = {
    "untuned": ModelSpec(name="LightGBM (untuned, re-run)", family="Boosting", folder=HERE / "untuned_rerun", scaled=False, config=m4.CONFIG,
                         fit=m4.fit, predict=capture(m4.predict, caps["untuned"]), save=m4.save, loss_name="sample-weighted binary log loss (objective='binary')",
                         history_ylabel="binary log loss (sample-weighted)"),
    "tuned": ModelSpec(name="LightGBM (tuned)", family="Boosting", folder=HERE / "tuned_model", scaled=False,
                       config={**TUNED_PARAMS, "max_rounds": T.CONFIG["max_rounds"], "early_stopping_rounds": T.CONFIG["early_stopping_rounds"],
                               "early_stop_val_share": T.CONFIG["early_stop_val_share"], "refit_on_all_rows": True, "seed": SEED, "source": "best_params.json"},
                       fit=tuned_fit, predict=capture(m4.predict, caps["tuned"]), save=m4.save, loss_name="sample-weighted binary log loss (objective='binary')",
                       history_ylabel="binary log loss (sample-weighted)"),
}
RES = {}
for name in ["untuned", "tuned"]:
    t0 = time.time(); RES[name] = run_experiment(specs[name]); print(f"{name}: harness run {time.time() - t0:.0f}s")

2026-11-18 17:48:48  === LightGBM (untuned, re-run) (Boosting) | seeds [42] | table raw ===


2026-11-18 17:48:48  python 3.13.7 on Windows-11-10.0.26300-SP0


C:\Users\Cathy\Desktop\hack 23\PATHSNAP\trained_models\model4\train_lightgbm.py:16: SyntaxWarning: invalid escape sequence '\.'
  Run from the trained_models folder:   ..\.venv\Scripts\python model4\train_lightgbm.py


2026-11-18 17:48:49  [train] 208,417 points, 1,943,052 candidate rows, miss rate 0.1037, fit_include rows 1,814,286


2026-11-18 17:48:49  [student] 19,160 points, 207,713 candidate rows, miss rate 0.0015, fit_include rows 207,713


    early stopping: best_iteration 102, val logloss 0.09694


... (24 outputs left out)

2026-11-18 17:54:47  seed 42 student: top1_all 0.8635  macroF1 0.9252  ll_norm 0.3829  continuity 0.7549  inference 16.6 ms / 1,000 points


2026-11-18 17:54:49  CV top1_all 0.7737 +/- 0.1289 | student top1_all 0.8635 | total 247.7s


2026-11-18 17:54:49  outputs written to C:\Users\Cathy\Desktop\hack 23\PATHSNAP\trained_models\model4\tuning\tuned_model


tuned: harness run 249s


In [7]:
"""Check that the re-run untuned model reproduces the stored model 4 results (it should: same code, same LightGBM version)."""
stored = json.loads((M4 / "results.json").read_text())
chk = pd.DataFrame({"stored model4": {"cv top1_all": stored["cv"]["all"]["top1_all"]["mean"], "cv top1_in_cands": stored["cv"]["all"]["top1_in_cands"]["mean"],
                                      "student top1_all": stored["student"]["all"]["top1_all"], "cv ll_norm": stored["cv"]["all"]["logloss_norm"]["mean"]},
                    "re-run here": {"cv top1_all": RES["untuned"]["cv"]["all"]["top1_all"]["mean"], "cv top1_in_cands": RES["untuned"]["cv"]["all"]["top1_in_cands"]["mean"],
                                    "student top1_all": RES["untuned"]["student"]["all"]["top1_all"], "cv ll_norm": RES["untuned"]["cv"]["all"]["logloss_norm"]["mean"]}})
chk["difference"] = chk["re-run here"] - chk["stored model4"]; display(chk.round(6))

,stored model4,re-run here,difference
cv top1_all,0.773289,0.773289,0.0
cv top1_in_cands,0.851447,0.851447,0.0
student top1_all,0.863570,0.863570,0.0
cv ll_norm,0.446065,0.446065,0.0


In [8]:
"""The untuned vs tuned table the report requires (all from the two harness runs)."""
def size_mb(folder): return (folder / "lightgbm_model.txt").stat().st_size / 1e6
def n_trees(folder): return sum(1 for line in open(folder / "lightgbm_model.txt", encoding="utf-8") if line.startswith("Tree="))
def row(r, folder):
    cv, st = r["cv"], r["student"]
    return {"CV Top-1 all (mean)": cv["all"]["top1_all"]["mean"], "CV Top-1 all (std)": cv["all"]["top1_all"]["std"],
            "CV Top-1 in-cands (mean)": cv["all"]["top1_in_cands"]["mean"], "CV Top-1 in-cands (std)": cv["all"]["top1_in_cands"]["std"],
            "CV Macro F1": cv["all"]["macro_f1"]["mean"], "CV log loss raw": cv["all"]["logloss_raw"]["mean"], "CV log loss norm": cv["all"]["logloss_norm"]["mean"],
            "CV ECE": cv["all"]["ece"]["mean"], "Student Top-1 all": st["all"]["top1_all"], "Student Top-1 clean": st["clean"]["top1_all"],
            "Student Macro F1": st["all"]["macro_f1"], "Student log loss raw": st["all"]["logloss_raw"], "Student log loss norm": st["all"]["logloss_norm"],
            "Route continuity (CV OOF)": r["route_continuity"]["cv_oof"]["pooled"], "Route continuity (student)": r["route_continuity"]["student"]["pooled"],
            "Train Top-1 all": r["train"]["all"]["top1_all"], "Final fit seconds": r["timing"]["final_fit_seconds"],
            "Inference ms / 1,000 points": r["timing"]["inference_ms_per_1000_points"], "Model size MB": size_mb(folder), "Trees": n_trees(folder)}
TABLE = pd.DataFrame({"untuned": row(RES["untuned"], HERE / "untuned_rerun"), "tuned": row(RES["tuned"], HERE / "tuned_model")})
TABLE["tuned - untuned"] = TABLE.tuned - TABLE.untuned
TABLE.to_csv(HERE / "untuned_vs_tuned.csv"); display(TABLE.round(5))

,untuned,tuned,tuned - untuned
CV Top-1 all (mean),0.77329,0.77366,0.00037
CV Top-1 all (std),0.12907,0.12889,-0.00018
CV Top-1 in-cands (mean),0.85145,0.85196,0.00051
CV Top-1 in-cands (std),0.03614,0.03746,0.00132
CV Macro F1,0.89262,0.89286,0.00024
CV log loss raw,0.08703,0.08561,-0.00142
CV log loss norm,0.44606,0.44076,-0.00530
CV ECE,0.01137,0.01009,-0.00128
Student Top-1 all,0.86357,0.86352,-0.00005
Student Top-1 clean,0.88482,0.88482,0.00000


In [9]:
"""Paired comparison: per fold, and per trip with a bootstrap confidence interval over trips (CV out-of-fold)."""
fu = pd.read_csv(HERE / "untuned_rerun" / "fold_results.csv"); ft = pd.read_csv(HERE / "tuned_model" / "fold_results.csv")
pf = fu[fu.subset == "all"].set_index("fold")[["top1_in_cands", "top1_all", "logloss_norm"]].join(ft[ft.subset == "all"].set_index("fold")[["top1_in_cands", "top1_all", "logloss_norm"]], lsuffix="_untuned", rsuffix="_tuned")
for m in ["top1_in_cands", "top1_all", "logloss_norm"]: pf[f"diff_{m}"] = pf[f"{m}_tuned"] - pf[f"{m}_untuned"]
display(pf.round(5)); print("folds where tuned Top-1 (in-cands) is higher:", int((pf.diff_top1_in_cands > 0).sum()), "of", len(pf))
def trips_of(folder, part):
    d = pd.read_csv(folder / "per_trip_top1.csv", dtype={"trip_id": str}); d = d[d.partition == part].set_index("trip_id")
    d["n_in"] = d.n_points * (1 - d.miss_rate); return d
pt = {}
for part in ["train_cv_oof", "student"]:
    u, t_ = trips_of(HERE / "untuned_rerun", part), trips_of(HERE / "tuned_model", part)
    j = u[["n_points", "n_in", "top1_in_cands", "top1_all"]].join(t_[["top1_in_cands", "top1_all"]], lsuffix="_u", rsuffix="_t")
    j["hits_in_u"] = j.top1_in_cands_u * j.n_in; j["hits_in_t"] = j.top1_in_cands_t * j.n_in
    pt[part] = j
def boot(j, B=10_000):
    """Bootstrap over TRIPS (resample trips with replacement): pooled Top-1 (in-cands) difference tuned - untuned."""
    idx = rng.integers(0, len(j), size=(B, len(j))); n = j.n_in.to_numpy()
    du = j.hits_in_u.to_numpy(); dt = j.hits_in_t.to_numpy()
    diffs = (dt[idx].sum(1) - du[idx].sum(1)) / n[idx].sum(1)
    point = (dt.sum() - du.sum()) / n.sum()
    return point, np.percentile(diffs, [2.5, 97.5])
BOOT = {}
for part in pt:
    point, ci = boot(pt[part]); BOOT[part] = {"diff": float(point), "ci95": [float(ci[0]), float(ci[1])], "n_trips": len(pt[part]),
                                             "trips_better": int((pt[part].top1_in_cands_t > pt[part].top1_in_cands_u).sum()), "trips_worse": int((pt[part].top1_in_cands_t < pt[part].top1_in_cands_u).sum())}
    print(f"{part}: pooled Top-1 (in-cands) difference {point:+.5f}, 95% bootstrap CI [{ci[0]:+.5f}, {ci[1]:+.5f}] over {len(pt[part])} trips | better {BOOT[part]['trips_better']}, worse {BOOT[part]['trips_worse']}")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].bar(pf.index.astype(str), pf.diff_top1_in_cands, color=np.where(pf.diff_top1_in_cands >= 0, "#2a9d8f", "#e76f51")); ax[0].axhline(0, c="k", lw=.8)
ax[0].set_title("Tuned - untuned CV Top-1 (in-cands) per fold"); ax[0].set_xlabel("fold")
d = (pt["train_cv_oof"].top1_in_cands_t - pt["train_cv_oof"].top1_in_cands_u).sort_values()
ax[1].bar(range(len(d)), d, color=np.where(d >= 0, "#2a9d8f", "#e76f51")); ax[1].axhline(0, c="k", lw=.8); ax[1].set_title("Tuned - untuned Top-1 (in-cands) per training trip (CV OOF)"); ax[1].set_xlabel("trips (sorted)")
plt.tight_layout(); plt.savefig(FIG / "paired_differences.png", dpi=140); plt.show()

,top1_in_cands_untuned,top1_all_untuned,logloss_norm_untuned,top1_in_cands_tuned,top1_all_tuned,logloss_norm_tuned,diff_top1_in_cands,diff_top1_all,diff_logloss_norm
fold,,,,,,,,,
0,0.82283,0.81847,0.54755,0.82077,0.81642,0.53730,-0.00206,-0.00205,-0.01024
1,0.90831,0.90664,0.27933,0.91006,0.90839,0.27289,0.00175,0.00175,-0.00643
2,0.83258,0.55968,0.49587,0.83403,0.56066,0.47607,0.00145,0.00098,-0.01980
3,0.86640,0.77398,0.43032,0.86872,0.77605,0.44634,0.00231,0.00207,0.01601
4,0.82711,0.80767,0.47726,0.82621,0.80679,0.47120,-0.00090,-0.00088,-0.00605


folds where tuned Top-1 (in-cands) is higher: 3 of 5
train_cv_oof: pooled Top-1 (in-cands) difference +0.00054, 95% bootstrap CI [-0.00149, +0.00269] over 80 trips | better 52, worse 24
student: pooled Top-1 (in-cands) difference -0.00005, 95% bootstrap CI [-0.00336, +0.00282] over 10 trips | better 7, worse 2


C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\3886104492.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "paired_differences.png", dpi=140); plt.show()


In [10]:
"""Over-fitting checks: train / CV / student gaps for both models, and best CV vs top-5 average (over-tuning the folds)."""
gap = pd.DataFrame({n: {"train Top-1 all": RES[n]["train"]["all"]["top1_all"], "CV Top-1 all": RES[n]["cv"]["all"]["top1_all"]["mean"],
                         "student Top-1 all": RES[n]["student"]["all"]["top1_all"], "train - CV": RES[n]["train"]["all"]["top1_all"] - RES[n]["cv"]["all"]["top1_all"]["mean"],
                         "CV - student": RES[n]["cv"]["all"]["top1_all"]["mean"] - RES[n]["student"]["all"]["top1_all"]} for n in RES})
display(gap.round(5))
print(f"best trial CV score {spread['best']:.5f} vs top-5 average {spread['top5_mean']:.5f}: difference {spread['best'] - spread['top5_mean']:.5f} (fold SE {spread['fold_se_of_best']:.5f})")

,untuned,tuned
train Top-1 all,0.79714,0.81564
CV Top-1 all,0.77329,0.77366
student Top-1 all,0.86357,0.86352
train - CV,0.02385,0.04198
CV - student,-0.09028,-0.08986


best trial CV score 0.85769 vs top-5 average 0.85729: difference 0.00041 (fold SE 0.01491)


In [11]:
"""Calibration: reliability curves of raw probabilities (CV out-of-fold and student) and normalised log loss, untuned vs tuned."""
train_rows = load_candidates("train", scaled=False); folds_sorted = sorted(set(train_rows.fold.tolist()))
def oof_from(cap):
    oof = np.full(train_rows.n, np.nan)
    for i, k in enumerate(folds_sorted): oof[train_rows.fold == k] = cap[i]
    assert not np.isnan(oof).any(); return oof
student_rows = load_candidates("student", scaled=False)
CAL = {}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for name, c in [("untuned", "#8d99ae"), ("tuned", "#2a6f97")]:
    oof = oof_from(caps[name]); ps = caps[name][-1]
    for j, (lab_, y_, p_) in enumerate([("CV out-of-fold", train_rows.y, oof), ("student-test", student_rows.y, ps)]):
        rt = reliability_table(y_, p_); ax[j].plot(rt.mean_pred, rt.frac_pos, "o-", c=c, label=f"{name} (ECE {expected_calibration_error(y_, np.clip(p_, 0, 1)):.4f})"); ax[j].set_title(lab_)
    CAL[name] = {"ece_cv_oof": expected_calibration_error(train_rows.y, np.clip(oof, 0, 1)), "ece_student": expected_calibration_error(student_rows.y, np.clip(ps, 0, 1)),
                 "logloss_norm_cv": RES[name]["cv"]["all"]["logloss_norm"]["mean"], "logloss_norm_student": RES[name]["student"]["all"]["logloss_norm"]}
for a in ax: a.plot([0, 1], [0, 1], "k--", lw=.8); a.set_xlabel("mean predicted probability"); a.set_ylabel("observed share of true arcs"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "calibration_untuned_vs_tuned.png", dpi=140); plt.show()
ECE_LIMIT = 0.02          # above this (on CV out-of-fold rows) we would test temperature / Platt scaling fitted inside the training folds
need_cal = CAL["tuned"]["ece_cv_oof"] > ECE_LIMIT
print({k: {kk: round(vv, 5) for kk, vv in v.items()} for k, v in CAL.items()}, "| calibration variant needed:", need_cal)

{'untuned': {'ece_cv_oof': 0.00759, 'ece_student': 0.00569, 'logloss_norm_cv': 0.44606, 'logloss_norm_student': 0.40177}, 'tuned': {'ece_cv_oof': 0.0063, 'ece_student': 0.00414, 'logloss_norm_cv': 0.44076, 'logloss_norm_student': 0.38289}} | calibration variant needed: False


C:\Users\Cathy\AppData\Local\Temp\ipykernel_28996\150731384.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "calibration_untuned_vs_tuned.png", dpi=140); plt.show()


### Findings

_filled in after the run_

## 5. Decision and the final model

**Rule (fixed before looking at the comparison):** use the tuned model only if (a) it is better on CV by a margin beyond the noise, meaning the 95%
bootstrap confidence interval over training trips of the pooled Top-1 (in-candidates) difference lies entirely above 0, **and** (b) its CV normalised
log loss is not worse than the untuned model's. Otherwise keep the untuned model. Student-test numbers are reported but do not enter the rule.

In [12]:
"""Apply the rule and write trained_models/final_model/ (model file, config, feature list) for the HMM and the API."""
ci_lo = BOOT["train_cv_oof"]["ci95"][0]
better_cv = ci_lo > 0
ll_ok = RES["tuned"]["cv"]["all"]["logloss_norm"]["mean"] <= RES["untuned"]["cv"]["all"]["logloss_norm"]["mean"]
use_tuned = bool(better_cv and ll_ok)
choice = "tuned" if use_tuned else "untuned"
src = (HERE / "tuned_model") if use_tuned else M4
FINAL.mkdir(exist_ok=True)
shutil.copy2(src / "lightgbm_model.txt", FINAL / "lightgbm_model.txt")
feat = feature_list(); (FINAL / "feature_list.json").write_text(json.dumps(feat, indent=2), encoding="utf-8")
params_final = TUNED_PARAMS if use_tuned else {**m4.PARAMS}
final_cfg = {"model": "LightGBM", "variant": choice, "model_file": "lightgbm_model.txt", "source_folder": str(src.relative_to(HERE.parents[2])).replace("\\", "/"),
             "params": params_final, "n_trees": n_trees(FINAL), "features": feat, "n_features": len(feat), "input_table": "raw candidate table (no scaling)",
             "output": "P(candidate arc is the true arc) per (point, candidate) row; normalise within each point for HMM emissions",
             "calibration": "none (raw probabilities are already well calibrated, see tuning/results.json)" if not need_cal else "see tuning notebook",
             "decision_rule": "tuned only if the 95% trip-bootstrap CI of the CV Top-1 (in-cands) gain is above 0 AND CV normalised log loss is not worse",
             "decision_inputs": {"ci95_cv_top1_in_cands_gain": BOOT["train_cv_oof"]["ci95"], "better_cv_beyond_noise": better_cv, "logloss_not_worse": ll_ok},
             "preprocessing_config": "data_preprocessing/data_preprocessing/preprocessing_config.json", "seed": SEED}
(FINAL / "config.json").write_text(json.dumps(final_cfg, indent=2), encoding="utf-8")
print("final choice:", choice, "| better on CV beyond noise:", better_cv, "| log loss not worse:", ll_ok)

In [ ]:
"""results.json for the tuning stage (every number used in the report answers)."""
summary = {"study": {"n_complete_trials": int(len(ok)), "n_trials_total": int(len(trials)), "sum_trial_minutes": float(total_s / 60), "study_wall_minutes": float(wall / 60),
                     "mean_trial_seconds": float(ok.seconds.mean()), "sampler": T.CONFIG["sampler"], "pruner": T.CONFIG["pruner"], "objective": T.CONFIG["objective"]},
           "spread_top5": spread, "chosen": frozen, "param_importance_fanova": {k: float(v) for k, v in imp.items()},
           "untuned_vs_tuned": {c: {k: float(v) for k, v in TABLE[c].items()} for c in TABLE.columns},
           "paired": {"per_fold": pf.reset_index().to_dict(orient="records"), "bootstrap_over_trips": BOOT},
           "overfitting": gap.to_dict(), "calibration": CAL, "calibration_variant_tested": bool(need_cal),
           "rerun_check_vs_stored_model4": chk.to_dict(), "decision": final_cfg["decision_inputs"] | {"final_choice": choice}}
(HERE / "results.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8"); print("results.json written")

In [ ]:
"""REPORT_ANSWERS.md: the report's six questions in simple words, with the untuned vs tuned table."""
P = best_params; U = TABLE.untuned; Tn = TABLE.tuned
fmt = lambda v: f"{v:.4f}" if abs(v) < 10 else f"{v:,.1f}"
tbl = "\n".join(["| Metric | Untuned | Tuned | Tuned - untuned |", "|---|---|---|---|"] + [f"| {k} | {fmt(U[k])} | {fmt(Tn[k])} | {TABLE.loc[k, 'tuned - untuned']:+.4f} |" for k in TABLE.index])
b = BOOT["train_cv_oof"]
txt = f"""# Bayesian hyperparameter optimisation: report answers

All numbers come from `tune_lightgbm.ipynb` (this folder): `optuna_trials.csv`, `results.json`, `untuned_vs_tuned.csv`.

**1. Which hyperparameters were searched?** Learning rate, number of leaves, maximum depth, minimum rows per leaf (`min_child_samples`), feature
fraction, bagging fraction, L1 and L2 regularisation, and minimum split gain. Class weighting and sample weights were kept exactly as in the untuned model.

**2. What ranges, and why?** Learning rate 0.01-0.2 (log), leaves 15-255 (log), depth -1 or 4-12, rows per leaf 10-300, feature and bagging
fraction 0.5-1.0, L1 and L2 0.001-10 (log), split gain 0-1. Each range brackets the untuned value and stays where training is practical: below 0.01
the learning rate needs more than the 2,000-round cap, below 0.5 a tree would often miss the main distance and heading features. Full reasons are in the notebook.

**3. What objective?** The mean 5-fold cross-validation Top-1 accuracy on points whose true arc is among the candidates, on the 80 training trips
with the stored trip-grouped folds. Points without their true arc among the candidates are wrong for every configuration, so leaving them out ranks
configurations the same way with less noise. Ties within one fold standard error go to the lower normalised log loss. Early stopping is nested inside
each training fold (20% of the training trips held out); the validation fold and student-test are never used to stop or choose.

**4. How many trials?** {len(ok)} unique complete configurations (46 complete evaluations, 6 of them accidental repeats) with Optuna's TPE sampler (seed 42), trial 0 being the untuned configuration. Total compute
{total_s/60:.0f} minutes ({ok.seconds.mean():.0f} s per trial on average).

**5. Best values?** Trial {chosen.number}: learning rate {P['learning_rate']:.4f}, leaves {P['num_leaves']}, max depth {P['max_depth']},
min rows per leaf {P['min_child_samples']}, feature fraction {P['feature_fraction']:.3f}, bagging fraction {P['bagging_fraction']:.3f},
L1 {P['lambda_l1']:.4g}, L2 {P['lambda_l2']:.4g}, min split gain {P['min_split_gain']:.4f}. CV Top-1 (in-candidates) {chosen.value:.4f};
the untuned configuration scored {base.value:.4f} in the same search. The top 5 trials span {spread['top5_range']:.4f}, against a fold standard error of {se:.4f}.

**6. Did the tuned model beat the untuned one?** With the full procedure (refit) on the same folds: CV Top-1 on all points {U['CV Top-1 all (mean)']:.4f} untuned
vs {Tn['CV Top-1 all (mean)']:.4f} tuned; in-candidates {U['CV Top-1 in-cands (mean)']:.4f} vs {Tn['CV Top-1 in-cands (mean)']:.4f}; normalised log loss
{U['CV log loss norm']:.4f} vs {Tn['CV log loss norm']:.4f}. Paired over the 80 training trips, the pooled gain is {b['diff']:+.4f} with a 95% bootstrap
interval of [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]. {'The interval is above zero and log loss is not worse, so the tuned model is used.' if use_tuned else 'This does not meet the rule (gain beyond the noise AND log loss not worse), so the untuned model is kept.'}
On student-test (used once, after freezing): Top-1 {U['Student Top-1 all']:.4f} untuned vs {Tn['Student Top-1 all']:.4f} tuned.

## Untuned vs tuned

{tbl}

Final model for the HMM and the API: **{choice}** LightGBM, saved in `trained_models/final_model/`.
"""
(HERE / "REPORT_ANSWERS.md").write_text(txt, encoding="utf-8"); print(txt)

### Findings

_filled in after the run_